# Modulo 08 - Gestione degli Errori

---

Ogni programmatore sbaglia, e ogni programma, prima o poi, incontra dati inaspettati. In questo modulo imparerai a **riconoscere**, **capire** e **gestire** i tre tipi di errore che compaiono in Python: gli **errori di sintassi**, che impediscono al codice di essere eseguito; gli **errori a tempo di esecuzione** (le eccezioni), che interrompono il programma a metà strada; e gli **errori di logica**, i più insidiosi, in cui il programma gira senza lamentarsi, ma restituisce il risultato sbagliato. Partendo dal `try/except` visto nel Modulo 03, approfondiremo con `else`, `raise`, la gerarchia delle eccezioni e le eccezioni personalizzate, tutto applicato a un problema reale: un'elaborazione giornaliera di fatture di un'azienda di telecomunicazioni che si rompe quando i dati arrivano fuori standard. Saper gestire gli errori è ciò che separa uno *script* che "funziona sulla mia macchina" da un programma pronto per la produzione.

Corso: Ready To Deploy

Creato da: [Enzo Schitini](https://www.linkedin.com/in/enzoschitini)

---

## Argomenti

| **Argomento** | Descrizione |
| --- | --- |
| 1. Tipi di errori | Panoramica dei tre tipi di errore in Python: sintassi, esecuzione e logica. |
| 2. Errori di sintassi | Perché il codice non arriva nemmeno a essere eseguito, gli errori di sintassi più comuni e come leggere il messaggio per correggerli. |
| 3. Errori a tempo di esecuzione | Eccezioni comuni, *traceback*, gerarchia delle eccezioni, `try/except/else/finally`, `raise` ed eccezioni personalizzate, applicati a un'elaborazione di fatture. |
| 4. Errori di logica | Come trovare errori che non generano alcun messaggio, usando `print()` e `assert`, inclusi i cicli infiniti. |

---

## 1. Tipi di errori

Prima di imparare a gestire un errore, bisogna sapere **di che tipo** è. Ogni tipo compare in un momento diverso e richiede una strategia diversa per essere risolto.

### 1.1 Definizione

| Tipo | Quando accade | Python avvisa? | Esempio | Come risolvere |
| --- | --- | --- | --- | --- |
| **Sintassi** | Prima dell'esecuzione, quando Python legge il codice | Sì, con `SyntaxError`, e **nessuna riga** viene eseguita | Dimenticare i due punti (`:`) di un `if` | Correggere il codice |
| **Esecuzione** (eccezione) | Durante l'esecuzione, in una riga specifica | Sì, con un'eccezione (`ZeroDivisionError`, `ValueError`...), e il programma si ferma **in quella riga** | Dividere per zero | Correggere il codice o **gestire** l'eccezione con `try/except` |
| **Logica** | Durante l'esecuzione | **No**: il programma gira fino alla fine | Calcolare una media con la formula sbagliata | Verificare i risultati e **debuggare** il codice |

**Errore di sintassi:** il codice è scritto in un modo che Python non capisce. Per questo la riga sotto è commentata: se venisse eseguita, l'intera cella fallirebbe prima ancora di iniziare.

In [1]:
eta = 19

# if eta >= 18    # ❌ SyntaxError: mancano i due punti alla fine
if eta >= 18:     # ✅ forma corretta
    print('Maggiorenne')

Maior de idade


**Errore a tempo di esecuzione:** il codice è scritto correttamente, ma un'operazione è impossibile con i valori ricevuti. Qui usiamo il `try/except`, visto nel Modulo 03, per mostrare l'errore senza bloccare il notebook.

In [2]:
try:
    print(1 / 0)
except ZeroDivisionError as errore:
    # type(errore).__name__ mostra il nome dell'eccezione
    print(f'{type(errore).__name__}: {errore}')

ZeroDivisionError: division by zero


**Errore di logica:** il codice gira senza nessun messaggio, ma il risultato è sbagliato. Qual è la media tra 8 e 10?

In [3]:
voto_1 = 8
voto_2 = 10

media = voto_1 + voto_2 / 2      # ❌ la divisione avviene prima della somma: 8 + 5
print(media)

media = (voto_1 + voto_2) / 2    # ✅ le parentesi garantiscono l'ordine corretto
print(media)

13.0
9.0


> ⚠️ **Attenzione:** l'errore di logica è il più pericoloso dei tre, proprio perché Python non si lamenta. Un report con un numero sbagliato può passare inosservato e portare a decisioni sbagliate.

---

## 2. Errori di sintassi

La **sintassi** è l'insieme delle regole di scrittura di un linguaggio, come la grammatica di una lingua. Quando una regola viene infranta, Python non riesce nemmeno a capire cosa deve fare, e per questo **non esegue nulla**.

### 2.1 Definizione

Prima di eseguire una cella, Python legge **tutto** il suo codice per capirlo. Se durante questa lettura trova un errore di sintassi, genera un `SyntaxError` e **nessuna riga viene eseguita**, nemmeno quelle che vengono prima dell'errore.

Per vedere questo accadere senza bloccare il notebook, useremo due funzioni native che ricevono un codice memorizzato in una *string*:

- `exec(codice)`: esegue il codice;
- `compile(codice, nome, 'exec')`: si limita ad **analizzare** il codice, senza eseguirlo.

Servono solo per questa dimostrazione. Nella pratica quotidiana scrivi il codice direttamente nella cella.

In [4]:
carrello_acquisti = [
    {'id': 3184, 'prezzo': 37.65, 'quantita': 10},
    {'id': 1203, 'prezzo': 81.20, 'quantita': 2},
    {'id': 8921, 'prezzo': 15.90, 'quantita': 2},
]

In [5]:
# Il print è PRIMA dell'errore di sintassi (manca ':' nel for)...
codice_con_errore = '''print('Questo messaggio viene prima dell\'errore')
for prodotto in carrello_acquisti
    print(prodotto)
'''

try:
    exec(codice_con_errore)
except SyntaxError as errore:
    # ...e ciononostante non viene mostrato: nulla è stato eseguito
    print(f'{type(errore).__name__}: {errore.msg} (riga {errore.lineno})')

SyntaxError: expected ':' (linha 2)


Nota che il messaggio `'Questo messaggio viene prima dell'errore'` **non è comparso**. Tieni a mente questa differenza: vedremo nella sezione 3 che, con gli errori di esecuzione, le righe precedenti all'errore vengono eseguite normalmente.

### 2.2 Errori di sintassi comuni

Per testare vari esempi, creiamo una funzione che analizza un codice e mostra l'errore di sintassi, indicando la riga e la posizione con una freccia (`^`), come fa Python stesso:

In [6]:
def verificare_sintassi(codice: str) -> None:
    '''Analizza il codice (senza eseguirlo) e mostra l'errore di sintassi, se presente.'''
    codice = codice.strip('\n')  # ignora righe vuote all'inizio e alla fine
    try:
        compile(codice, '<esempio>', 'exec')
    except SyntaxError as errore:
        riga_con_errore = codice.splitlines()[errore.lineno - 1]
        print(f'{type(errore).__name__} alla riga {errore.lineno}: {errore.msg}')
        print(f'    {riga_con_errore}')
        if errore.offset:
            print('    ' + ' ' * (errore.offset - 1) + '^')
    else:
        print('Nessun errore di sintassi trovato.')

**Esempio:** dimenticare i due punti (`:`) alla fine di un `for`, `if`, `elif`, `else` o `def`.

In [7]:
verificare_sintassi('''
for prodotto in carrello_acquisti
    print(prodotto)
''')

SyntaxError na linha 1: expected ':'
    for produto in carrinho_compras
                                   ^


**Esempio:** mettere una condizione nell'`else`. L'`else` significa "in tutti gli altri casi", quindi **non riceve mai** una condizione. Per testare un'altra condizione, usa `elif`.

In [8]:
verificare_sintassi('''
for prodotto in carrello_acquisti:
    if prodotto['id'] == 3184:
        print('Prodotto 3184')
    else prodotto['id'] == 1203:
        print('Prodotto 1203')
''')

SyntaxError na linha 4: expected ':'
        else produto['id'] == 1203:
             ^


In [9]:
# ✅ Forma corretta: elif per la seconda condizione
for prodotto in carrello_acquisti:
    if prodotto['id'] == 3184:
        print('Prodotto 3184')
    elif prodotto['id'] == 1203:
        print('Prodotto 1203')

Produto 3184
Produto 1203


**Esempio:** indentazione errata. In Python, gli spazi all'inizio della riga definiscono quali righe appartengono a un blocco. Questo errore ha perfino un nome proprio: `IndentationError`.

In [10]:
verificare_sintassi('''
for prodotto in carrello_acquisti:
print(prodotto)
''')

IndentationError na linha 2: expected an indented block after 'for' statement on line 1
    print(produto)
    ^


**Esempio:** dimenticare di chiudere parentesi, parentesi quadre o virgolette.

In [11]:
verificare_sintassi('''
print('Totale prodotti:', len(carrello_acquisti)
''')

SyntaxError na linha 1: '(' was never closed
    print('Total de produtos:', len(carrinho_compras)
         ^


In [12]:
verificare_sintassi('''
messaggio = 'Benvenuto su Ready To Deploy
''')

SyntaxError na linha 1: unterminated string literal (detected at line 1)
    mensagem = 'Bem-vindo ao Ready To Deploy
               ^


**Esempio:** usare `=` (assegnazione) dove dovrebbe esserci `==` (confronto).

In [13]:
verificare_sintassi('''
if len(carrello_acquisti) = 3:
    print('Carrello con 3 prodotti')
''')

SyntaxError na linha 1: cannot assign to function call here. Maybe you meant '==' instead of '='?
    if len(carrinho_compras) = 3:
       ^


**Esempio:** usare `return` fuori da una funzione. Il `return` ha senso solo dentro un `def`.

In [14]:
verificare_sintassi('''
eta = 19
if eta > 18:
    return True
''')

SyntaxError na linha 3: 'return' outside function
        return True
        ^


Riassumendo i casi più frequenti:

| Errore | Esempio errato | Correzione |
| --- | --- | --- |
| Mancano i due punti | `for prodotto in carrello` | `for prodotto in carrello:` |
| Condizione nell'`else` | `else prezzo > 10:` | `elif prezzo > 10:` |
| Indentazione errata | blocco del `for` senza rientro | rientrare il blocco con 4 spazi |
| Parentesi o virgolette non chiuse | `print('Ciao'` | `print('Ciao')` |
| `=` al posto di `==` | `if totale = 3:` | `if totale == 3:` |
| `return` fuori da una funzione | `return True` isolato nel codice | usare `return` solo dentro un `def` |

### 2.3 Come correggere

Gli errori di sintassi **non possono essere gestiti** con `try/except` all'interno della stessa cella: dato che Python non riesce nemmeno a leggere il codice, il `try` non arriva mai a essere eseguito. L'unica via d'uscita è **correggere il codice**. Per farlo, leggi con attenzione il messaggio di errore:

1. **Tipo di errore:** `SyntaxError` o `IndentationError`;
2. **Riga:** dove Python ha notato il problema;
3. **Freccia `^`:** la posizione approssimativa dell'errore nella riga;
4. **Descrizione:** nelle versioni più recenti di Python è molto specifica, come `expected ':'` ("atteso `:`").

> 💡 **Suggerimento:** Python indica il punto in cui **ha notato** l'errore, che non sempre è dove esso **si trova** davvero. Una parentesi dimenticata in una riga viene spesso segnalata solo nella riga successiva. Se la riga indicata sembra corretta, controlla anche la riga precedente.

> 💡 **Suggerimento:** editor come Colab e VS Code sottolineano in rosso buona parte degli errori di sintassi mentre digiti. Presta attenzione a questi avvisi prima di eseguire la cella.

---

## 3. Errori a tempo di esecuzione

Un codice con la sintassi perfetta può comunque fallire: basta ricevere un dato inaspettato. Questi errori, chiamati **eccezioni**, sono i più comuni nella pratica quotidiana, ed è qui che la gestione degli errori fa più differenza.

### 3.1 Motivazione

Lavori come analista di dati in un'azienda di telecomunicazioni e devi comunicare al team vendite **quanto l'azienda incasserà questo mese**. Ogni giorno, il team di ingegneria invia un file CSV con le fatture dei clienti:

| Colonna | Significato |
| --- | --- |
| `customerID` | Identificativo del cliente |
| `PaymentMethod` | Metodo di pagamento |
| `MonthlyCharges` | Importo della fattura mensile |
| `TotalCharges` | Importo totale già pagato dal cliente |
| `Churn` | Il cliente ha cancellato il servizio? |

Prima di tutto, creiamo i file di esempio. Il file del **giorno 1** è arrivato nel formato concordato:

In [15]:
def creare_file(nome_file: str, contenuto: str) -> None:
    '''Crea (o sovrascrive) un file di testo con il contenuto indicato.'''
    with open(nome_file, mode='w', encoding='utf-8') as file:
        file.write(contenuto)


creare_file('telecom_giorno_01.csv', '''customerID,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7010-BRBUU,Credit card (automatic),24.1,1734.65,No
9688-YGXVR,Credit card (automatic),88.15,3973.2,No
9286-DOJGF,Bank transfer (automatic),74.95,2869.85,Yes
6994-KERXL,Electronic check,55.9,238.5,No
2181-UAESM,Electronic check,53.45,119.5,No
4312-GVYNH,Bank transfer (automatic),49.85,3370.2,No
2495-KZNFB,Electronic check,90.65,2989.6,No
4367-NHWMM,Mailed check,24.9,24.9,No
8898-KASCD,Mailed check,35.55,1309.15,No
''')

Questa è la funzione che hai scritto per sommare le fatture. Legge il file riga per riga e prende il valore della **3ª colonna** (`MonthlyCharges`, indice `2`):

In [16]:
def sommare_fatture(nome_file: str) -> float:
    fatture = []

    with open(nome_file, mode='r', encoding='utf-8') as file:
        file.readline()  # scarta l'intestazione
        for riga in file:
            colonne = riga.strip().split(',')
            fattura = float(colonne[2])  # 3ª colonna: MonthlyCharges
            fatture.append(fattura)

    return sum(fatture)


totale_da_incassare = sommare_fatture('telecom_giorno_01.csv')
print(f'Totale da incassare: {totale_da_incassare:.2f}')

Total a receber: 497.50


Tutto ok il giorno 1. Ma il **giorno 2**, l'ingegneria ha cambiato l'ordine delle colonne senza avvisare: `MonthlyCharges` ora è la 2ª colonna, e la 3ª è diventata `PaymentMethod`.

In [17]:
creare_file('telecom_giorno_02.csv', '''customerID,MonthlyCharges,PaymentMethod,TotalCharges,Churn
7010-BRBUU,24.1,Credit card (automatic),1734.65,No
9688-YGXVR,88.15,Credit card (automatic),3973.2,No
9286-DOJGF,74.95,Bank transfer (automatic),2869.85,Yes
6994-KERXL,55.9,Electronic check,238.5,No
2181-UAESM,53.45,Electronic check,119.5,No
4312-GVYNH,49.85,Bank transfer (automatic),3370.2,No
2495-KZNFB,90.65,Electronic check,2989.6,No
4367-NHWMM,24.9,Mailed check,24.9,No
8898-KASCD,35.55,Mailed check,1309.15,No
''')

Elaborando il nuovo file, il programma **si rompe**. Senza il `try/except` sotto (che usiamo solo per non bloccare il notebook), l'esecuzione si fermerebbe qui con un `ValueError`:

In [18]:
try:
    totale_da_incassare = sommare_fatture('telecom_giorno_02.csv')
    print(f'Totale da incassare: {totale_da_incassare:.2f}')
except ValueError as errore:
    print(f'{type(errore).__name__}: {errore}')

ValueError: could not convert string to float: 'Credit card (automatic)'


Python ha provato a convertire il testo `'Credit card (automatic)'` in numero, il che è impossibile.

**Come possiamo far sì che l'elaborazione gestisca questo tipo di problema, avvisando chiaramente cosa è andato storto o, ancora meglio, continuando a funzionare anche con le colonne scambiate?**

### 3.2 Definizione

Un **errore a tempo di esecuzione** accade mentre il programma è in esecuzione. Il codice viene eseguito normalmente **fino alla riga dell'errore**. In quel momento, Python "lancia" (o "solleva") un'**eccezione** e, se nessuno la gestisce, il programma viene interrotto.

Guarda la differenza rispetto all'errore di sintassi: qui, il messaggio prima dell'errore **viene** mostrato.

In [19]:
try:
    print('Questo messaggio viene prima dell\'errore')  # viene eseguito
    print(10 / 0)                                       # qui viene sollevata l'eccezione
    print('Questo messaggio viene dopo l\'errore')       # non viene mai eseguito
except ZeroDivisionError as errore:
    print(f'{type(errore).__name__}: {errore}')

Esta mensagem vem antes do erro
ZeroDivisionError: division by zero


Ogni tipo di problema genera un'eccezione con un nome diverso. Queste sono le più comuni:

| Eccezione | Quando accade | Esempio |
| --- | --- | --- |
| `ZeroDivisionError` | Divisione per zero | `10 / 0` |
| `TypeError` | Operazione con tipi incompatibili | `'eta: ' + 30` |
| `ValueError` | Tipo corretto, ma valore impossibile da usare | `float('abc')` |
| `IndexError` | Posizione inesistente in una lista | `[1, 2, 3][5]` |
| `KeyError` | Chiave inesistente in un dizionario | `{'a': 1}['b']` |
| `NameError` | Variabile o funzione che non esiste | `print(variabile_inesistente)` |
| `AttributeError` | Metodo o attributo che l'oggetto non ha | `'testo'.sommare()` |
| `FileNotFoundError` | File che non esiste | `open('non_esiste.csv')` |

**Esempio:** operazione numerica impossibile, dividendo un conto tra zero persone.

In [20]:
importo_conto = 132.85
numero_persone = 0

try:
    importo_per_persona = importo_conto / numero_persone
except ZeroDivisionError as errore:
    print(f'{type(errore).__name__}: {errore}')

ZeroDivisionError: float division by zero


**Esempio:** combinazione di tipi incompatibili, concatenando testo con numero.

In [21]:
nome = 'André Perez'
eta = 30

try:
    presentazione = 'Mi chiamo ' + nome + ' e ho ' + eta + ' anni.'
except TypeError as errore:
    print(f'{type(errore).__name__}: {errore}')

# ✅ Con la f-string, la conversione in testo è automatica
print(f'Mi chiamo {nome} e ho {eta} anni.')

TypeError: can only concatenate str (not "int") to str
Meu nome é André Perez e eu tenho 30 anos.


**Esempio:** accedere a una posizione che non esiste in una lista.

In [22]:
anni = [2019, 2020, 2021]

try:
    anno_corrente = anni[3]  # le posizioni valide sono 0, 1 e 2
except IndexError as errore:
    print(f'{type(errore).__name__}: {errore}')

IndexError: list index out of range


**Esempio:** accedere a una chiave che non esiste in un dizionario.

In [23]:
corsi = {
    'python': {'nome': 'Python per l\'Analisi dei Dati', 'durata_mesi': 2.5},
    'sql': {'nome': 'SQL per l\'Analisi dei Dati', 'durata_mesi': 2},
}

print(corsi['python'])
print(corsi['sql'])

try:
    print(corsi['analista'])
except KeyError as errore:
    print(f'{type(errore).__name__}: {errore}')

{'nome': 'Python para Análise de Dados', 'duracao_meses': 2.5}
{'nome': 'SQL para Análise de Dados', 'duracao_meses': 2}
KeyError: 'analista'


> 💡 **Suggerimento:** spesso si può **evitare** l'eccezione invece di gestirla. Per i dizionari, il metodo `.get(chiave, valore_predefinito)` restituisce il valore predefinito quando la chiave non esiste, senza generare errore.

In [24]:
corso = corsi.get('analista', 'Corso non trovato')
print(corso)

Curso não encontrado


### 3.3 Leggere il *traceback*

Quando un'eccezione non viene gestita, Python mostra il ***traceback***: la "traccia" delle chiamate di funzione che hanno portato all'errore. Sembra spaventoso, ma segue sempre la stessa struttura. Il modulo `traceback` permette di mostrarlo anche quando catturiamo l'eccezione:

In [25]:
import traceback

def calcolare_importo_per_persona(importo_totale: float, numero_persone: int) -> float:
    return importo_totale / numero_persone

def chiudere_conto(importo_totale: float, numero_persone: int) -> None:
    importo = calcolare_importo_per_persona(importo_totale, numero_persone)
    print(f'Ogni persona paga {importo:.2f}')

try:
    chiudere_conto(132.85, 0)
except ZeroDivisionError:
    traceback.print_exc()  # mostra il traceback completo, come se l'errore non fosse gestito

Traceback (most recent call last):
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 11, in <module>
    fechar_conta(132.85, 0)
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 7, in fechar_conta
    valor = calcular_valor_por_pessoa(valor_total, quantidade_pessoas)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\schit\AppData\Local\Temp\ipykernel_37644\1307365568.py", line 4, in calcular_valor_por_pessoa
    return valor_total / quantidade_pessoas
           ~~~~~~~~~~~~^~~~~~~~~~~~~~~~~~~~
ZeroDivisionError: float division by zero


Leggi il *traceback* **dal basso verso l'alto**:

1. **Ultima riga:** il tipo dell'eccezione e il messaggio (`ZeroDivisionError: division by zero`). Comincia sempre da qui;
2. **Righe subito sopra:** il file, il numero di riga e la funzione dove è accaduto l'errore (`calcolare_importo_per_persona`);
3. **Righe più in alto:** il percorso di chiamate fino a lì. Chi ha chiamato chi (`chiudere_conto` ha chiamato `calcolare_importo_per_persona`).

> 💡 **Suggerimento:** quando cerchi un errore su internet, copia l'**ultima riga** del *traceback*. È la parte più utile e di solito porta direttamente a spiegazioni e soluzioni.

### 3.4 Gerarchia delle eccezioni

Nel Modulo 06 abbiamo visto che le classi possono **ereditare** da altre classi. Le eccezioni di Python sono proprio classi organizzate in una gerarchia di ereditarietà. Un estratto semplificato:

```
Exception
 ├── ArithmeticError
 │    └── ZeroDivisionError
 ├── LookupError
 │    ├── IndexError
 │    └── KeyError
 ├── ValueError
 ├── TypeError
 └── OSError
      └── FileNotFoundError
```

Un `except` cattura l'eccezione indicata **e tutte quelle che ereditano da essa**. La funzione nativa `issubclass()` conferma queste relazioni:

In [26]:
print(issubclass(ZeroDivisionError, ArithmeticError))  # True
print(issubclass(IndexError, LookupError))             # True
print(issubclass(KeyError, LookupError))               # True
print(issubclass(ValueError, Exception))               # True: quasi tutto eredita da Exception

True
True
True
True


**Esempio:** poiché `IndexError` e `KeyError` ereditano da `LookupError`, un singolo `except LookupError` gestisce entrambi i casi.

In [27]:
def cercare(collezione, chiave_o_posizione):
    try:
        return collezione[chiave_o_posizione]
    except LookupError as errore:
        print(f'Non trovato ({type(errore).__name__}): {errore}')
        return None

cercare(anni, 10)             # lista: genera IndexError
cercare(corsi, 'analista')    # dizionario: genera KeyError

Não encontrado (IndexError): list index out of range
Não encontrado (KeyError): 'analista'


> ⚠️ **Attenzione:** l'**ordine degli `except` conta**. Python usa il **primo** che corrisponde all'eccezione. Per questo, metti sempre le eccezioni **più specifiche prima** e quelle più generiche (come `Exception`) per ultime. Un `except Exception` in cima catturerebbe tutto, e gli `except` sotto non verrebbero mai usati.

> ⚠️ **Attenzione:** evita di catturare `Exception` (o usare un `except:` senza nulla) solo per "far sparire l'errore". Questo nasconde problemi reali, inclusi errori di battitura nel tuo stesso codice. Cattura solo le eccezioni che sai gestire.

### 3.5 try / except / else / finally

Nel Modulo 03 abbiamo visto `try`, `except` e `finally`. La struttura completa ha anche il blocco `else`:

```python
try:
    # codice che può generare un'eccezione
except TipoDiEccezione as errore:
    # eseguito SE l'eccezione accade
else:
    # eseguito SE NESSUNA eccezione accade
finally:
    # eseguito SEMPRE, con o senza eccezione
```

| Blocco | Obbligatorio? | Quando viene eseguito |
| --- | --- | --- |
| `try` | Sì | Sempre, fino alla prima eccezione |
| `except` | Almeno un `except` o un `finally` | Solo se si verifica l'eccezione indicata |
| `else` | No | Solo se **nessuna** eccezione si verifica nel `try` |
| `finally` | No | Sempre, alla fine, qualunque cosa accada |

**Esempio:** consultando un anno in una collezione.

In [28]:
def consultare_anno(anni, posizione: int) -> None:
    try:
        anno = anni[posizione]
    except IndexError:
        print(f'Posizione non valida. Scegli un valore tra 0 e {len(anni) - 1}.')
    except TypeError as errore:
        print(f'Collezione non supportata: {errore}')
    else:
        print(f'Anno trovato: {anno}')
    finally:
        print('Consultazione terminata.')
        print('-' * 30)

In [29]:
lista_anni = [2019, 2020, 2021]
insieme_anni = {2019, 2020, 2021}

consultare_anno(lista_anni, 1)     # senza errore: esegue l'else
consultare_anno(lista_anni, 3)     # IndexError
consultare_anno(insieme_anni, 0)   # TypeError: gli insiemi non hanno posizione

Ano encontrado: 2020
Consulta finalizada.
------------------------------
Posição inválida. Escolha um valor entre 0 e 2.
Consulta finalizada.
------------------------------
Coleção não suportada: 'set' object is not subscriptable
Consulta finalizada.
------------------------------


> 💡 **Suggerimento:** perché usare l'`else` invece di mettere tutto dentro il `try`? Per mantenere il `try` con il **minimo** di codice possibile: solo la riga che può fallire. Così, un'eccezione inaspettata in un'altra parte del codice non viene catturata per sbaglio dall'`except`.

### 3.6 Sollevare eccezioni con `raise`

Finora abbiamo solo **reagito** alle eccezioni di Python. Con la parola chiave `raise`, possiamo noi stessi **sollevare** un'eccezione quando rileviamo una situazione non valida, interrompendo la funzione e avvisando chi l'ha chiamata.

**Esempio:** validando i dati prima di dividere un conto.

In [30]:
def dividere_conto(importo_totale: float, numero_persone: int) -> float:
    if numero_persone <= 0:
        raise ValueError(f'Il numero di persone deve essere maggiore di zero (ricevuto: {numero_persone}).')
    return importo_totale / numero_persone

In [31]:
print(dividere_conto(132.85, 5))

try:
    print(dividere_conto(132.85, -2))
except ValueError as errore:
    print(f'Non è stato possibile dividere il conto: {errore}')

26.57
Não foi possível dividir a conta: A quantidade de pessoas deve ser maior que zero (recebido: -2).


> 💡 **Suggerimento:** senza la validazione, `dividere_conto(132.85, -2)` restituirebbe un valore negativo senza lamentarsi, un **errore di logica**. Sollevare l'eccezione trasforma un problema silenzioso in un errore chiaro e facile da trovare.

Dentro un `except`, possiamo anche **rilanciare** l'eccezione in avanti con un `raise` da solo. Questo è utile quando la funzione vuole registrare il problema, ma lasciare la decisione su cosa fare a chi l'ha chiamata:

In [32]:
def convertire_valore(testo: str) -> float:
    try:
        return float(testo)
    except ValueError:
        print(f"[registro] valore non valido ricevuto: '{testo}'")
        raise  # rilancia la STESSA eccezione a chi ha chiamato


try:
    convertire_valore('Credit card (automatic)')
except ValueError as errore:
    print(f'Chi ha chiamato ha ricevuto l\'errore: {errore}')

[registro] valor inválido recebido: 'Credit card (automatic)'
Quem chamou recebeu o erro: could not convert string to float: 'Credit card (automatic)'


Infine, possiamo **sostituire** l'eccezione con un'altra più chiara per il contesto, mantenendo quella originale come causa, con `raise NuovaEccezione(...) from errore`:

In [33]:
def ottenere_anno_corrente(anni: list) -> int:
    try:
        return anni[3]
    except IndexError as errore:
        raise ValueError(f'La lista di anni deve avere almeno 4 elementi (ne ha {len(anni)}).') from errore


try:
    ottenere_anno_corrente([2019, 2020, 2021])
except ValueError as errore:
    print(f'Errore: {errore}')
    print(f'Causa:  {type(errore.__cause__).__name__}: {errore.__cause__}')  # l'eccezione originale

Erro:  A lista de anos precisa ter pelo menos 4 elementos (tem 3).
Causa: IndexError: list index out of range


> 💡 **Suggerimento:** il `from errore` conserva l'eccezione originale nel *traceback*, che passa a mostrarle entrambe: "l'eccezione sopra è stata la causa diretta dell'eccezione sotto". Questo facilita molto l'indagine del problema.

### 3.7 Eccezioni personalizzate

Poiché le eccezioni sono classi, possiamo creare le **nostre**, ereditando da `Exception` (o da un'eccezione più specifica). Un'eccezione con nome proprio chiarisce **cosa** è andato storto e permette a chi chiama la funzione di gestire quel caso separatamente dagli altri.

In [34]:
class FatturaNonValidaError(Exception):
    '''Sollevata quando un file di fatture non può essere elaborato.'''


try:
    raise FatturaNonValidaError('Riga 5: valore della fattura vuoto.')
except FatturaNonValidaError as errore:
    print(f'{type(errore).__name__}: {errore}')

FaturaInvalidaError: Linha 5: valor da fatura em branco.


> 💡 **Suggerimento:** per convenzione, il nome di un'eccezione termina con `Error`, come le eccezioni native (`ValueError`, `KeyError`). La *docstring* da sola è già sufficiente come corpo della classe, senza bisogno di `pass`.

### 3.8 Rivisitando la motivazione

Torniamo all'elaborazione delle fatture. Possiamo migliorarla su due fronti:

1. **Robustezza:** invece di affidarci alla posizione fissa della colonna, localizziamo la colonna `MonthlyCharges` **per nome**, leggendo l'intestazione. Così, scambiare l'ordine delle colonne smette di essere un problema;
2. **Messaggi chiari:** se comunque un dato risulta non valido (la colonna non esiste, un valore è vuoto...), solleviamo una `FatturaNonValidaError` che dice **in quale riga** e **perché**, conservando l'eccezione originale con `from`.

La funzione nativa `enumerate()` percorre le righe del file e, allo stesso tempo, le conta. Con `start=2`, il conteggio inizia da 2, perché la riga 1 è l'intestazione.

In [35]:
def sommare_fatture(nome_file: str, colonna: str = 'MonthlyCharges') -> float:
    '''Somma i valori della colonna indicata. Solleva FatturaNonValidaError se qualche dato non è valido.'''
    fatture = []

    with open(nome_file, mode='r', encoding='utf-8') as file:
        intestazione = file.readline().strip().split(',')

        # 1. Localizza la colonna per nome (list.index solleva ValueError se non esiste)
        try:
            posizione_colonna = intestazione.index(colonna)
        except ValueError as errore:
            raise FatturaNonValidaError(f"Colonna '{colonna}' non trovata nell'intestazione.") from errore

        # 2. Converte il valore di ogni riga, indicando la riga esatta in caso di errore
        for numero_riga, riga in enumerate(file, start=2):
            valore = riga.strip().split(',')[posizione_colonna]
            try:
                fattura = float(valore)
            except ValueError as errore:
                raise FatturaNonValidaError(f"Riga {numero_riga}: valore non valido '{valore}'.") from errore
            else:
                fatture.append(fattura)

    return sum(fatture)

Ora entrambi i file vengono elaborati correttamente, e danno lo stesso totale, dato che è cambiato solo l'ordine delle colonne:

In [36]:
for nome_file in ['telecom_giorno_01.csv', 'telecom_giorno_02.csv']:
    totale_da_incassare = sommare_fatture(nome_file)
    print(f'{nome_file}: totale da incassare = {totale_da_incassare:.2f}')

telecom_dia_01.csv: total a receber = 497.50
telecom_dia_02.csv: total a receber = 497.50


E se, il **giorno 3**, un cliente arrivasse con la fattura vuota? La funzione non ha modo di indovinare il valore corretto, quindi **non deve** nascondere il problema. Invece, solleva un errore chiaro. Chi chiama la funzione decide cosa fare: qui, avvisiamo il team di ingegneria invece di inviare un totale sbagliato al team vendite.

In [37]:
creare_file('telecom_giorno_03.csv', '''customerID,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7010-BRBUU,Credit card (automatic),24.1,1734.65,No
9688-YGXVR,Credit card (automatic),88.15,3973.2,No
9286-DOJGF,Bank transfer (automatic),,2869.85,Yes
6994-KERXL,Electronic check,55.9,238.5,No
''')

In [38]:
for nome_file in ['telecom_giorno_01.csv', 'telecom_giorno_02.csv', 'telecom_giorno_03.csv']:
    try:
        totale_da_incassare = sommare_fatture(nome_file)
    except FatturaNonValidaError as errore:
        print(f'❌ {nome_file}: file respinto. {errore} Avvisa il team di ingegneria.')
    except FileNotFoundError:
        print(f'❌ {nome_file}: file non trovato.')
    else:
        print(f'✅ {nome_file}: totale da incassare = {totale_da_incassare:.2f}')

✅ telecom_dia_01.csv: total a receber = 497.50
✅ telecom_dia_02.csv: total a receber = 497.50
❌ telecom_dia_03.csv: arquivo rejeitado. Linha 4: valor inválido ''. Avise o time de engenharia.


Confronta con la versione iniziale: prima, una modifica nel file mandava in crash il programma con un messaggio generico (`could not convert string to float`). Ora, lo scambio delle colonne viene risolto automaticamente, e un dato davvero non valido genera un messaggio che indica **il file**, **la riga** e **il valore** con il problema. Nel frattempo, gli altri file continuano a essere elaborati.

> 💡 **Suggerimento:** nel Modulo 05 abbiamo sommato liste con `reduce()`. Qui usiamo la funzione nativa `sum()`, che fa la stessa cosa in modo più diretto e leggibile.

---

## 4. Errori di logica

Un errore di logica non genera nessuna eccezione: il programma gira fino alla fine e restituisce un risultato, solo che **sbagliato**. Poiché Python non indica il problema, sta a noi diffidare dei risultati e indagare il codice.

### 4.1 Definizione

Un **errore di logica** accade quando il codice fa esattamente ciò che è stato scritto, ma ciò che è stato scritto non è ciò che volevamo. Le cause più comuni sono:

- Formule sbagliate o con l'ordine delle operazioni scambiato (come la media della sezione 1);
- Limiti sbagliati in cicli e slice, saltando o ripetendo elementi;
- Condizioni invertite (`>` al posto di `<`, `and` al posto di `or`);
- Cicli che non finiscono mai (**cicli infiniti**).

Lo strumento principale per trovarli è il **debug**: mostrare risultati intermedi per scoprire in quale punto il valore calcolato smette di essere quello atteso.

### 4.2 Limiti delle collezioni

**Esempio:** calcolando il valore totale del carrello della spesa. Il risultato atteso è **570,70**: $37{,}65 \times 10 + 81{,}20 \times 2 + 15{,}90 \times 2$.

In [39]:
valore_totale = 0

for indice in range(1, len(carrello_acquisti)):
    prodotto = carrello_acquisti[indice]
    valore_totale += prodotto['prezzo'] * prodotto['quantita']

print(f'Valore totale: {valore_totale:.2f}')

Valor total: 194.20


Il codice è girato senza nessun errore, ma il valore è molto più basso di quanto atteso. Qualcosa è sbagliato nella **logica**.

### 4.3 Debug con `print()`

Il modo più semplice per fare debug è mettere un `print()` dentro il ciclo per **vedere** cosa sta succedendo a ogni passo:

In [40]:
valore_totale = 0

for indice in range(1, len(carrello_acquisti)):
    prodotto = carrello_acquisti[indice]
    print(f'[debug] indice {indice}: prodotto {prodotto["id"]}')  # print temporaneo
    valore_totale += prodotto['prezzo'] * prodotto['quantita']

print(f'Valore totale: {valore_totale:.2f}')

[depuração] índice 1: produto 1203
[depuração] índice 2: produto 8921
Valor total: 194.20


Ecco, il problema è emerso: il ciclo inizia all'indice `1` e **salta il primo prodotto** (indice `0`, il `3184`). Il `range(1, ...)` dovrebbe essere `range(0, ...)`. Ancora meglio: percorrendo la lista direttamente con `for prodotto in carrello_acquisti`, non c'è nessun indice su cui sbagliare:

In [41]:
valore_totale = 0

for prodotto in carrello_acquisti:
    valore_totale += prodotto['prezzo'] * prodotto['quantita']

print(f'Valore totale: {valore_totale:.2f}')

Valor total: 570.70


> 💡 **Suggerimento:** dopo aver trovato l'errore, **rimuovi** i `print()` di debug, per non inquinare l'output del programma.

> 💡 **Suggerimento:** per proteggerti da errori di logica, puoi verificare se un risultato ha senso con `assert condizione, 'messaggio'`. Se la condizione è falsa, Python solleva un `AssertionError` con il messaggio. Se è vera, non succede nulla.

In [42]:
quantita_prodotti = len(carrello_acquisti)
prodotti_sommati = 0

for prodotto in carrello_acquisti:
    prodotti_sommati += 1

# Se qualche prodotto fosse stato saltato, questa riga solleverebbe un AssertionError
assert prodotti_sommati == quantita_prodotti, 'Qualche prodotto non è stato sommato!'
print('Verifica OK: tutti i prodotti sono stati sommati.')

Conferência OK: todos os produtos foram somados.


### 4.4 Cicli infiniti

Un tipo speciale di errore di logica è il **ciclo infinito**: un ciclo la cui condizione di arresto non viene mai raggiunta, lasciando il programma "bloccato" per sempre.

È più comune con il ciclo `while` ("finché"), che ripete un blocco **finché** una condizione è vera. A differenza del `for`, che percorre una collezione con una fine definita, il `while` dipende dal fatto che qualcosa al suo interno **cambi la condizione** a un certo punto.

In [43]:
tentativo = 1

while tentativo <= 3:
    print(f'Tentativo di connessione {tentativo}...')
    tentativo += 1  # senza questa riga, tentativo sarebbe sempre 1 e il ciclo non finirebbe mai

print('Fine dei tentativi.')

Tentativa de conexão 1...
Tentativa de conexão 2...
Tentativa de conexão 3...
Fim das tentativas.


Se la riga `tentativo += 1` fosse dimenticata, la condizione `tentativo <= 3` sarebbe **sempre** vera e il ciclo girerebbe per sempre. Per questo è così importante. Un altro modo per garantire l'uscita è usare il `break`, visto nel Modulo 03, per interrompere il ciclo quando una condizione viene raggiunta:

In [44]:
contatore = 0

while True:           # la condizione è sempre vera...
    contatore += 1
    if contatore > 10:
        break         # ...quindi il break è l'UNICA uscita dal ciclo

print(f'Il ciclo è stato eseguito {contatore - 1} volte.')

O laço executou 10 vezes.


> ⚠️ **Attenzione:** se una cella continua a essere eseguita molto più a lungo di quanto ti aspetti, sospetta un ciclo infinito. In Colab, interrompi l'esecuzione cliccando sul pulsante di **stop** accanto alla cella (o con la scorciatoia `Ctrl + M` + `I`). Un ciclo infinito che accumula dati in memoria può persino far bloccare l'intera sessione.

---

## Riepilogo del Modulo

| Tipo di errore | Quando accade | Segnale | Come gestirlo |
| --- | --- | --- | --- |
| Sintassi | Prima dell'esecuzione | `SyntaxError` / `IndentationError`, nulla viene eseguito | Leggere il messaggio (riga e `^`) e correggere il codice |
| Esecuzione | Durante l'esecuzione | Eccezione (`ValueError`, `KeyError`...) e *traceback* | Correggere, evitare (validare i dati, `.get()`) o gestire con `try/except` |
| Logica | Durante l'esecuzione | Nessuno: solo il risultato sbagliato | Verificare i risultati, fare debug con `print()` e usare `assert` |

| Risorsa | A cosa serve |
| --- | --- |
| `try` / `except` | Cattura e gestisce un'eccezione |
| `else` | Viene eseguito solo se non c'è stata eccezione nel `try` |
| `finally` | Viene eseguito sempre, con o senza eccezione |
| `raise Eccezione('messaggio')` | Solleva un'eccezione |
| `raise` (da solo, dentro l'`except`) | Rilancia l'eccezione corrente a chi ha chiamato |
| `raise NuovaEccezione(...) from errore` | Sostituisce l'eccezione, conservando quella originale come causa |
| `class MioErrore(Exception):` | Crea un'eccezione personalizzata |
| `traceback.print_exc()` | Mostra il *traceback* di un'eccezione catturata |
| `assert condizione, 'messaggio'` | Verifica se un risultato ha senso |

Buone pratiche: mantieni il `try` piccolo, cattura eccezioni **specifiche** (dalle più specifiche alle più generiche), non "ingoiare" mai gli errori senza gestirli e preferisci messaggi che dicano **dove** e **perché** qualcosa è fallito.